In [1]:
# calculadora.py

import argparse
import sys
from huggingface_hub import login
import os
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig
from trl import SFTTrainer


In [2]:


def upload_blob_from_json(bucket_name, json_content, destination_blob_name):
    """Uploads a string to the bucket."""

    from google.cloud import storage
    import json
    
    storage_client = storage.Client()
    bucket = storage_client.get_bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    text_content = json.dumps(json_content)
    blob.upload_from_string(text_content)



In [3]:
job_id         = 'job1'
experiment_id  = 'exp1'
run_id         = 'run1'
location       = 'us-east4'
experiment_metadata_gspath  = 'gs://genai-dev-tmp/experiment_metadata'
project_id     = 'xx'

if experiment_metadata_gspath.startswith('gs://'):
    experiment_metadata_gspath = experiment_metadata_gspath[5:]

experiment_metadata_bucket = experiment_metadata_gspath.split('/')[0]
experiment_metadata_path   = '/'.join(experiment_metadata_gspath.split('/')[1:])
metrics_file_name    = f'{experiment_metadata_path}/metrics.json'

log_dir = '/tmp'

if log_dir.strip()=='':
    log_dir = "logs"

epochs  = 2
per_device_train_batch_size = 1
gradient_accumulation_steps = 2
max_steps = 10

print(f"""
--------------- env vars --------------------
LOCATION           {location}
GCLOUD_PROJECT_ID  {project_id}
EXPERIMENT_ID      {experiment_id}
RUN_ID             {run_id}
JOB_ID             {job_id}
EXPERIMENT_METADATA_BUCKET  {experiment_metadata_bucket}
EXPERIMENT_METADATA_GSPATH  {experiment_metadata_gspath}

--------------- args --------------------
epocs                       {epochs}
per_device_train_batch_size {per_device_train_batch_size}
gradient_accumulation_steps {gradient_accumulation_steps}

-------------- experiment data --------------
metrics_file_name      {metrics_file_name}
log dir                {log_dir}
""")

# -------------------------------------------------------
# -------------------------------------------------------




--------------- env vars --------------------
LOCATION           us-east4
GCLOUD_PROJECT_ID  xx
EXPERIMENT_ID      exp1
RUN_ID             run1
JOB_ID             job1
EXPERIMENT_METADATA_BUCKET  genai-dev-tmp
EXPERIMENT_METADATA_GSPATH  genai-dev-tmp/experiment_metadata

--------------- args --------------------
epocs                       2
per_device_train_batch_size 1
gradient_accumulation_steps 2

-------------- experiment data --------------
metrics_file_name      experiment_metadata/metrics.json
log dir                /tmp



In [4]:



hf_token = os.environ['HF_TOKEN']
login(hf_token)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:

# System message for the assistant
system_message = """You are a text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA."""

# User prompt that combines the user query and the schema
user_prompt = """Given the <USER_QUERY> and the <SCHEMA>, generate the corresponding SQL command to retrieve the desired data, considering the query's syntax, semantics, and schema constraints.

<SCHEMA>
{context}
</SCHEMA>

<USER_QUERY>
{question}
</USER_QUERY>
"""
def create_conversation(sample):
  return {
    "messages": [
      # {"role": "system", "content": system_message},
      {"role": "user", "content": user_prompt.format(question=sample["sql_prompt"], context=sample["sql_context"])},
      {"role": "assistant", "content": sample["sql"]}
    ]
  }

# Load dataset from the hub
dataset = load_dataset("philschmid/gretel-synthetic-text-to-sql", split="train")
dataset = dataset.shuffle().select(range(12500))

# Convert dataset to OAI messages
dataset = dataset.map(create_conversation, remove_columns=dataset.features,batched=False)
# split dataset into 10,000 training samples and 2,500 test samples
dataset = dataset.train_test_split(test_size=2500/12500)

# Print formatted user prompt
print(dataset["train"][345]["messages"][1]["content"])



# Hugging Face model id
model_id = "google/gemma-3-1b-pt" # or `google/gemma-3-4b-pt`, `google/gemma-3-12b-pt`, `google/gemma-3-27b-pt`

# Select model class based on id
if model_id == "google/gemma-3-1b-pt":
    model_class = AutoModelForCausalLM
else:
    model_class = AutoModelForImageTextToText

# Check if GPU benefits from bfloat16
if torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

# Define model init arguments
model_kwargs = dict(
    attn_implementation="eager", # Use "flash_attention_2" when running on Ampere or newer GPU
    torch_dtype=torch_dtype, # What torch dtype to use, defaults to auto
    device_map="auto", # Let torch decide how to load the model
)

# BitsAndBytesConfig: Enables 4-bit quantization to reduce model size/memory usage
model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=model_kwargs['torch_dtype'],
    bnb_4bit_quant_storage=model_kwargs['torch_dtype'],
)

# Load model and tokenizer
model = model_class.from_pretrained(model_id, **model_kwargs)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it") # Load the Instruction Tokenizer to use the official Gemma template


# setup training

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"] # make sure to save the lm_head and embed_tokens as you train the special tokens
)

args = SFTConfig(
    output_dir="gemma-text-to-sql",         # directory to save and repository id
    max_length=512,                         # max sequence length for model and packing of the dataset
    packing=True,                           # Groups multiple samples in the dataset into a single sequence
    num_train_epochs=epochs,                                    # number of training epochs
    per_device_train_batch_size=per_device_train_batch_size,    # batch size per device during training
    gradient_accumulation_steps=per_device_train_batch_size,    # number of steps before performing a backward/update pass
    max_steps = max_steps,
    gradient_checkpointing=True,            # use gradient checkpointing to save memory
    optim="adamw_torch_fused",              # use fused adamw optimizer
    logging_dir=log_dir,
    logging_steps=10,                       # log every 10 steps
    save_strategy="epoch",                  # save checkpoint every epoch
    learning_rate=2e-4,                     # learning rate, based on QLoRA paper
    fp16=True if torch_dtype == torch.float16 else False,   # use float16 precision
    bf16=True if torch_dtype == torch.bfloat16 else False,   # use bfloat16 precision
    max_grad_norm=0.3,                      # max gradient norm based on QLoRA paper
    warmup_ratio=0.03,                      # warmup ratio based on QLoRA paper
    lr_scheduler_type="constant",           # use constant learning rate scheduler
    push_to_hub=False,                      # push model to hub
    report_to="tensorboard",                # report metrics
    dataset_kwargs={
        "add_special_tokens": False, # We template with special tokens
        "append_concat_token": True, # Add EOS token as separator token between examples
    },
)



Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

SELECT MIN(visitor_age) FROM Family_Friendly_Exhibitions WHERE city = 'London' AND year = 2021;


In [6]:
# Create Trainer object
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    peft_config=peft_config,
    processing_class=tokenizer
)




Tokenizing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [14]:
train_output = trainer.train()

Step,Training Loss
10,0.471600


In [16]:
train_output

TrainOutput(global_step=10, training_loss=0.47159528732299805, metrics={'train_runtime': 31.2845, 'train_samples_per_second': 0.32, 'train_steps_per_second': 0.32, 'total_flos': 30412261721088.0, 'train_loss': 0.47159528732299805, 'epoch': 0.0022888532845044634})

In [21]:
train_output.metrics

{'train_runtime': 31.2845,
 'train_samples_per_second': 0.32,
 'train_steps_per_second': 0.32,
 'total_flos': 30412261721088.0,
 'train_loss': 0.47159528732299805,
 'epoch': 0.0022888532845044634}